In [ ]:
import os, sys, subprocess
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"
os.environ["WANDB_MODE"] = "disabled"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "ultralytics==8.3.221", "albumentations==2.0.8", "pycocotools==2.0.10",
    "roboflow==1.2.11", "PyYAML==6.0.2"])



In [ ]:
# ===== CHANGE THESE CONTROLS =====
MODELS = "yolo"               
AUGMENTATIONS = ["none", "default", "custom"]  
PHOTO_SETS = ["Normal"]        
DISPLAY_CONFIDENCE = 0.25      # Picture display only; does not change mAP scoring.


RUNS_ROOT = "/kaggle/input"
OWN_DATA_ROOT = "/kaggle/input/datasets/senomynearl/owndata-panopticon"
OUTPUT_BASE = "/kaggle/working/controlled_evaluation"


SKIP_SIZE_MISMATCH = True
SKIP_MISSING_SEGMENTATION = True
REUSE_EXISTING_RESULTS = True
REUSE_LEGACY_RESULTS = True    


In [ ]:
from pathlib import Path
import sys, importlib
CODE_DIR = Path("/kaggle/working/seg046_code")
CODE_DIR.mkdir(parents=True, exist_ok=True)
(CODE_DIR / "seg046.py").write_text('"""NRP 046: Kaggle instance-segmentation training and held-out evaluation.\nSource-of-truth labels are COCO instance masks; all model outputs use COCOeval.\n"""\nfrom __future__ import annotations\nimport contextlib, csv, hashlib, io, json, os, random, shutil, subprocess, sys, time\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\nimport numpy as np\nfrom PIL import Image, ImageDraw\n\nSEED = 46\nMODELS = (\'yolo11n-seg\', \'yolo11s-seg\', \'maskrcnn\')\nREGIMES = (\'none\', \'default\', \'custom\')\nDATASET_URL = \'https://universe.roboflow.com/panopticindoor/panoptic-indoor-segmentation/dataset/5\'\nEXPECTED_COUNTS = {\'train\': 397, \'valid\': 166, \'test\': 359}\nYOLO_AUG = {\n    \'none\': dict(hsv_h=0., hsv_s=0., hsv_v=0., degrees=0., translate=0., scale=0.,\n                 shear=0., perspective=0., flipud=0., fliplr=0., mosaic=0., mixup=0.,\n                 copy_paste=0., cutmix=0., bgr=0., close_mosaic=0),\n    \'default\': dict(hsv_h=.015, hsv_s=.7, hsv_v=.4, degrees=0., translate=.1, scale=.5,\n                    shear=0., perspective=0., flipud=0., fliplr=.5, mosaic=1., mixup=0.,\n                    copy_paste=0., cutmix=0., bgr=0., close_mosaic=10),\n    \'custom\': dict(hsv_h=.01, hsv_s=.02, hsv_v=.02, degrees=10., translate=.02, scale=.02,\n                   shear=0., perspective=0., flipud=0., fliplr=0., mosaic=0.,\n                   mixup=0., copy_paste=0., cutmix=0., bgr=0., close_mosaic=0),\n}\n\ndef write_json(path, data):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + \'.tmp\')\n    tmp.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding=\'utf-8\')\n    tmp.replace(path)\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for b in iter(lambda: f.read(1024 * 1024), b\'\'): h.update(b)\n    return h.hexdigest()\n\ndef seed_all(seed=SEED):\n    import torch\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n\ndef seed_worker(worker_id):\n    import torch\n    s = torch.initial_seed() % 2**32\n    random.seed(s); np.random.seed(s)\n\ndef require_gpu():\n    import torch, torchvision\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'Enable a GPU accelerator in Kaggle Settings before training.\')\n    # Also fail early if Kaggle torch/torchvision are incompatible.\n    torchvision.ops.nms(torch.tensor([[0.,0.,1.,1.]],device=\'cuda\'),\n                        torch.tensor([1.],device=\'cuda\'), .5)\n    print(\'GPU:\', torch.cuda.get_device_name(0), \'| torch:\', torch.__version__,\n          \'| torchvision:\', torchvision.__version__)\n\ndef obtain_dataset(data_root=\'\', download_root=\'/kaggle/working/panoptic_v5\'):\n    if data_root:\n        root = Path(data_root)\n    else:\n        root = Path(download_root)\n        if not list(root.rglob(\'_annotations.coco.json\')):\n            from kaggle_secrets import UserSecretsClient\n            from roboflow import Roboflow\n            try:\n                key = UserSecretsClient().get_secret(\'ROBOFLOW_API_KEY\')\n            except Exception as exc:\n                raise RuntimeError(\'Add the Kaggle secret ROBOFLOW_API_KEY and enable it for this notebook, \'\n                                   \'or set DATA_ROOT to an attached COCO Segmentation v5 export.\') from exc\n            # Suppress SDK output: download URLs can contain the API key.\n            try:\n                with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):\n                    ds = (Roboflow(api_key=key).workspace(\'panopticindoor\')\n                          .project(\'panoptic-indoor-segmentation\').version(5)\n                          .download(\'coco-segmentation\', location=str(root), overwrite=False))\n                root = Path(ds.location)\n            except Exception:\n                raise RuntimeError(\'Roboflow download failed. Check Internet and the API-key permission, \'\n                                   \'or attach a manual v5 COCO Segmentation export as DATA_ROOT.\') from None\n    if not root.exists(): raise FileNotFoundError(root)\n    # Accommodate one extra directory level in a Kaggle dataset upload.\n    candidates = [p.parent.parent for p in root.rglob(\'_annotations.coco.json\') if p.parent.name==\'train\']\n    if (root/\'train/_annotations.coco.json\').exists(): return root\n    candidates = sorted(set(candidates))\n    if len(candidates) != 1:\n        raise ValueError(f\'Expected one COCO training export under {root}; found {len(candidates)}. Set DATA_ROOT precisely.\')\n    return candidates[0]\n\ndef decode_segmentation(seg, height, width):\n    from pycocotools import mask as mu\n    if isinstance(seg, list):\n        if not seg: raise ValueError(\'Missing instance polygon; a bounding-box export cannot train segmentation.\')\n        if any(len(p)<6 or len(p)%2 for p in seg): raise ValueError(\'Invalid COCO polygon.\')\n        rle = mu.merge(mu.frPyObjects(seg, height, width))\n    elif isinstance(seg, dict):\n        rle = mu.frPyObjects(seg, height, width) if isinstance(seg.get(\'counts\'), list) else dict(seg)\n        if isinstance(rle.get(\'counts\'), str): rle[\'counts\'] = rle[\'counts\'].encode()\n    else:\n        raise ValueError(\'Expected COCO instance polygons/RLE, not a semantic/panoptic PNG export.\')\n    m = mu.decode(rle)\n    if m.ndim == 3: m = np.any(m, axis=2)\n    if m.shape != (height, width): raise ValueError(\'Mask/image dimensions differ.\')\n    return m.astype(np.uint8)\n\ndef encode_mask(mask):\n    from pycocotools import mask as mu\n    rle = mu.encode(np.asfortranarray(mask.astype(np.uint8)))\n    rle[\'counts\'] = rle[\'counts\'].decode(\'ascii\')\n    return rle\n\ndef category_names(raw):\n    used = {a[\'category_id\'] for a in raw[\'annotations\']}\n    parents = {c.get(\'supercategory\') for c in raw[\'categories\']}\n    # Roboflow often adds an unused parent category, which is not a trainable class.\n    cats = [c for c in raw[\'categories\'] if c[\'id\'] in used or c[\'name\'] not in parents]\n    names = [c[\'name\'] for c in cats]\n    if len(names)!=len(set(names)): raise ValueError(\'Duplicate class names in COCO categories.\')\n    return sorted(names)\n\ndef find_image(json_path, record):\n    base = Path(json_path).parent\n    p = base / record[\'file_name\']\n    if not p.is_file(): p = base / Path(record[\'file_name\']).name\n    if not p.is_file(): raise FileNotFoundError(f\'Missing image {record["file_name"]} near {json_path}\')\n    return p.resolve()\n\ndef canonicalize(json_path, names):\n    from pycocotools import mask as mu\n    raw = read_json(json_path)\n    if not raw.get(\'images\'): raise ValueError(f\'Empty image split: {json_path}\')\n    if not raw.get(\'annotations\'): raise ValueError(f\'No annotations: {json_path}\')\n    ids = [x[\'id\'] for x in raw[\'images\']]\n    if len(ids)!=len(set(ids)): raise ValueError(\'Duplicate image IDs.\')\n    cat = {c[\'id\']:c[\'name\'] for c in raw[\'categories\']}\n    name_to_id = {n:i+1 for i,n in enumerate(names)}\n    images, paths, hashes = [], {}, {}\n    for rec in raw[\'images\']:\n        p = find_image(json_path, rec)\n        with Image.open(p) as im: w,h = im.size\n        if (w,h)!=(rec[\'width\'],rec[\'height\']):\n            raise ValueError(f\'Image size differs from COCO metadata: {p.name}. Re-export without changing the images.\')\n        images.append(dict(id=rec[\'id\'],file_name=str(p),width=w,height=h))\n        paths[rec[\'id\']] = (w,h)\n        hashes[rec[\'id\']] = sha256(p)\n    anns=[]\n    for ann in raw[\'annotations\']:\n        if ann[\'image_id\'] not in paths: raise ValueError(\'Annotation references an absent image.\')\n        name=cat[ann[\'category_id\']]\n        if name not in name_to_id:\n            raise ValueError(f\'Unknown class {name!r}. Match the training class spelling exactly; do not silently merge labels.\')\n        w,h=paths[ann[\'image_id\']]\n        m=decode_segmentation(ann.get(\'segmentation\'),h,w)\n        rle=mu.encode(np.asfortranarray(m))\n        area=float(mu.area(rle)); box=mu.toBbox(rle).tolist()\n        if area<=0: raise ValueError(f\'Empty instance mask, annotation {ann["id"]}. Repair the source annotation.\')\n        anns.append(dict(id=len(anns)+1,image_id=ann[\'image_id\'],category_id=name_to_id[name],\n                         segmentation=ann[\'segmentation\'],bbox=box,area=area,iscrowd=int(ann.get(\'iscrowd\',0))))\n    return dict(info={\'description\':\'Canonical instance masks; IDs remapped by exact class name\'},\n                licenses=raw.get(\'licenses\',[]),images=images,annotations=anns,\n                categories=[dict(id=i+1,name=n) for i,n in enumerate(names)]), hashes\n\ndef prepare_data(source_root, prepared_root, strict_v5=True):\n    root=Path(source_root); out=Path(prepared_root); out.mkdir(parents=True,exist_ok=True)\n    sources={s:root/s/\'_annotations.coco.json\' for s in (\'train\',\'valid\',\'test\')}\n    if not sources[\'valid\'].exists(): sources[\'valid\']=root/\'val/_annotations.coco.json\'\n    for p in sources.values():\n        if not p.exists(): raise FileNotFoundError(f\'Missing original split: {p}; do not substitute validation for test.\')\n    names=category_names(read_json(sources[\'train\']))\n    prepared={}; split_hashes={}; counts={}; class_counts={}; duplicate_pairs=[]\n    for s,p in sources.items():\n        canonical, hashes=canonicalize(p,names)\n        counts[s]=len(canonical[\'images\']); split_hashes[s]=set(hashes.values())\n        prepared[s]=str(out/f\'{s}.json\'); write_json(prepared[s],canonical)\n        class_counts[s]=dict(Counter(names[a[\'category_id\']-1] for a in canonical[\'annotations\']))\n    if strict_v5 and counts != EXPECTED_COUNTS:\n        raise ValueError(f\'Expected v5 split sizes {EXPECTED_COUNTS}, found {counts}. Check version/export; \'\n                         \'only set STRICT_V5=False for an explicitly documented exception.\')\n    for a,b in [(\'train\',\'valid\'),(\'train\',\'test\'),(\'valid\',\'test\')]:\n        overlap=split_hashes[a]&split_hashes[b]\n        if overlap: duplicate_pairs.append({\'splits\':[a,b],\'exact_duplicate_count\':len(overlap)})\n    # Preserve the assignment splits but expose any source duplication, never resplit silently.\n    if duplicate_pairs: print(\'WARNING: source split duplicates; discuss leakage in the report:\',duplicate_pairs)\n    fingerprint=hashlib.sha256(json.dumps({\'sources\':{s:sha256(p) for s,p in sources.items()},\n        \'image_hashes\':{s:sorted(h) for s,h in split_hashes.items()},\'classes\':names},sort_keys=True).encode()).hexdigest()\n    report=dict(dataset_url=DATASET_URL,counts=counts,classes=names,class_counts=class_counts,\n                exact_cross_split_duplicates=duplicate_pairs,dataset_fingerprint=fingerprint,\n                source_json_hashes={s:sha256(p) for s,p in sources.items()})\n    write_json(out/\'dataset_report.json\',report); write_json(out/\'classes.json\',names)\n    print(\'Classes:\',len(names),\'| original splits:\',counts)\n    return dict(root=str(out),names=names,paths=prepared,fingerprint=fingerprint)\n\ndef coco_to_yolo(prepared):\n    """Convert polygons only for YOLO training. Evaluate on untouched COCO masks."""\n    import yaml\n    from ultralytics.data.converter import merge_multi_segment\n    root=Path(prepared[\'root\'])/\'yolo\'; multi=0\n    for split,json_path in prepared[\'paths\'].items():\n        raw=read_json(json_path); by=defaultdict(list)\n        for a in raw[\'annotations\']:\n            if not a[\'iscrowd\']: by[a[\'image_id\']].append(a)\n        imgdir=root/\'images\'/split; labdir=root/\'labels\'/split\n        imgdir.mkdir(parents=True,exist_ok=True); labdir.mkdir(parents=True,exist_ok=True)\n        for rec in raw[\'images\']:\n            source=Path(rec[\'file_name\']); dest=imgdir/f\'{rec["id"]}{source.suffix.lower()}\'\n            if not dest.exists(): dest.symlink_to(source)\n            lines=[]\n            for a in by[rec[\'id\']]:\n                seg=a[\'segmentation\']\n                if not isinstance(seg,list):\n                    raise ValueError(\'YOLO conversion needs polygon export. RLE masks cannot be losslessly \'\n                                     \'represented in YOLO polygon labels. Re-export as COCO Segmentation polygons.\')\n                if len(seg)>1:\n                    points=np.concatenate(merge_multi_segment(seg),axis=0); multi+=1\n                else: points=np.asarray(seg[0],dtype=float).reshape(-1,2)\n                points=points / np.array([rec[\'width\'],rec[\'height\']])\n                if not np.isfinite(points).all(): raise ValueError(\'Non-finite polygon coordinates.\')\n                if (points < -1e-5).any() or (points > 1.00001).any():\n                    raise ValueError(\'Out-of-image polygon: repair source labels before training.\')\n                points=np.clip(points,0,1)\n                lines.append(str(a[\'category_id\']-1)+\' \'+\' \'.join(f\'{v:.7f}\' for v in points.ravel()))\n            (labdir/f\'{rec["id"]}.txt\').write_text(\'\\n\'.join(lines),encoding=\'utf-8\')\n    config={\'path\':str(root.resolve()),\'train\':\'images/train\',\'val\':\'images/valid\',\'test\':\'images/test\',\'names\':prepared[\'names\']}\n    (root/\'data.yaml\').write_text(yaml.safe_dump(config,sort_keys=False),encoding=\'utf-8\')\n    write_json(root/\'conversion_report.json\',{\'multipolygon_instances_bridged\':multi,\n        \'note\':\'YOLO uses connected polygons for multipart objects; COCO evaluation retains the original masks. Crowd labels excluded from training.\'})\n    return str(root/\'data.yaml\')\n\nclass IdentityAlbumentations:\n    def __init__(self,*args,**kwargs): self.transform=None; self.p=0.\n    def __call__(self,labels): return labels\n\n@contextlib.contextmanager\ndef yolo_augmentation_policy(regime):\n    import ultralytics.data.augment as aug\n    original=aug.Albumentations\n    # Native default includes four p=.01 photometric Albumentations operations.\n    # Disable the hidden photometric pipeline for none and custom.\n    if regime!=\'default\': aug.Albumentations=IdentityAlbumentations\n    else:\n        import albumentations  # fail instead of silently changing the default policy\n        if original().transform is None: raise RuntimeError(\'Default Albumentations failed to initialize.\')\n    try: yield\n    finally: aug.Albumentations=original\n\ndef collect_environment(run_dir):\n    import torch, torchvision\n    write_json(Path(run_dir)/\'environment.json\',dict(python=sys.version,torch=torch.__version__,\n        torchvision=torchvision.__version__,gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else \'CPU\',\n        cuda=torch.version.cuda,seed=SEED))\n    p=subprocess.run([sys.executable,\'-m\',\'pip\',\'freeze\'],capture_output=True,text=True)\n    (Path(run_dir)/\'pip_freeze.txt\').write_text(p.stdout,encoding=\'utf-8\')\n\ndef plot_predictions(rec, predictions, categories, out, threshold=.25):\n    im=Image.open(rec[\'file_name\']).convert(\'RGB\'); overlay=Image.new(\'RGBA\',im.size,(0,0,0,0)); draw=ImageDraw.Draw(overlay)\n    palette=[(255,90,70),(50,200,255),(170,90,255),(255,210,40),(40,225,130)]\n    for p in predictions:\n        if p[\'score\']<threshold: continue\n        m=decode_segmentation(p[\'segmentation\'],rec[\'height\'],rec[\'width\'])\n        color=palette[(p[\'category_id\']-1)%len(palette)]\n        rgba=Image.new(\'RGBA\',im.size,color+(95,))\n        overlay.alpha_composite(Image.composite(rgba,Image.new(\'RGBA\',im.size),Image.fromarray(m*255)))\n        x,y,w,h=p[\'bbox\']; draw=ImageDraw.Draw(overlay)\n        draw.rectangle([x,y,x+w,y+h],outline=color+(255,),width=2)\n        draw.text((x,max(0,y-12)),f\'{categories[p["category_id"]]} {p["score"]:.2f}\',fill=color+(255,))\n    Image.alpha_composite(im.convert(\'RGBA\'),overlay).convert(\'RGB\').save(out)\n\ndef evaluate_coco(json_path,predictions,out_dir,visual_limit=12):\n    from pycocotools.coco import COCO\n    from pycocotools.cocoeval import COCOeval\n    out=Path(out_dir); out.mkdir(parents=True,exist_ok=True)\n    gt=COCO(str(json_path)); write_json(out/\'predictions_coco.json\',predictions)\n    # Remove bbox for segmentation loadRes so size-area bins use mask area.\n    dt=gt.loadRes([{k:v for k,v in p.items() if k!=\'bbox\'} for p in predictions]) if predictions else COCO()\n    if not predictions:\n        dt.dataset=dict(images=gt.dataset[\'images\'],categories=gt.dataset[\'categories\'],annotations=[]);dt.createIndex()\n    ev=COCOeval(gt,dt,\'segm\'); ev.params.maxDets=[1,10,100]\n    log=io.StringIO()\n    with contextlib.redirect_stdout(log): ev.evaluate();ev.accumulate();ev.summarize()\n    print(log.getvalue()); (out/\'coco_summary.txt\').write_text(log.getvalue())\n    metrics={k:float(ev.stats[i]) for i,k in enumerate([\'mask_AP50_95\',\'mask_AP50\',\'mask_AP75\',\n        \'mask_AP_small\',\'mask_AP_medium\',\'mask_AP_large\',\'mask_AR1\',\'mask_AR10\',\'mask_AR100\',\n        \'mask_AR_small\',\'mask_AR_medium\',\'mask_AR_large\'])}\n    metrics.update(images=len(gt.imgs),instances=len(gt.anns),max_detections=100)\n    write_json(out/\'metrics.json\',metrics)\n    with open(out/\'per_class_AP.csv\',\'w\',newline=\'\') as f:\n        writer=csv.writer(f); writer.writerow([\'category_id\',\'class\',\'mask_AP50_95\',\'mask_AP50\',\'gt_instances\'])\n        for k,cid in enumerate(ev.params.catIds):\n            all_ap=ev.eval[\'precision\'][:,:,k,0,-1]; ap50=ev.eval[\'precision\'][0,:,k,0,-1]\n            writer.writerow([cid,gt.cats[cid][\'name\'],float(all_ap[all_ap>=0].mean()) if (all_ap>=0).any() else \'\',\n                             float(ap50[ap50>=0].mean()) if (ap50>=0).any() else \'\',len(gt.getAnnIds(catIds=[cid]))])\n    if visual_limit:\n        vis=out/\'overlays\';vis.mkdir(exist_ok=True); by=defaultdict(list)\n        for p in predictions: by[p[\'image_id\']].append(p)\n        for rec in list(gt.imgs.values())[:visual_limit]:\n            plot_predictions(rec,by[rec[\'id\']],{k:v[\'name\'] for k,v in gt.cats.items()},vis/f\'{rec["id"]}.jpg\')\n    return metrics\n\ndef predict_yolo(weights,json_path,imgsz=640):\n    from ultralytics import YOLO\n    from pycocotools import mask as mu\n    model=YOLO(str(weights)); predictions=[]\n    for rec in read_json(json_path)[\'images\']:\n        r=model.predict(rec[\'file_name\'],imgsz=imgsz,device=0,conf=.001,iou=.7,\n                        max_det=100,retina_masks=True,verbose=False)[0]\n        if r.masks is None: continue\n        for mask,score,cls in zip(r.masks.data.cpu().numpy(),r.boxes.conf.cpu().numpy(),r.boxes.cls.cpu().numpy()):\n            if mask.shape!=(rec[\'height\'],rec[\'width\']): raise RuntimeError(\'YOLO mask was not restored to source resolution.\')\n            m=(mask>.5).astype(np.uint8); enc=encode_mask(m)\n            predictions.append(dict(image_id=rec[\'id\'],category_id=int(cls)+1,score=float(score),\n                                    segmentation=enc,bbox=mu.toBbox(enc).tolist()))\n    return predictions\n\n# Torch imports are local except the Dataset base; notebook installs dependencies before importing.\nimport torch\nfrom torch.utils.data import Dataset,DataLoader\n\ndef custom_hsv(image,cfg):\n    """Match Ultralytics 8.3.221 HSV ranges; operate on RGB image only."""\n    import cv2\n    gains=np.random.uniform(-1,1,3)*[cfg[\'hsv_h\'],cfg[\'hsv_s\'],cfg[\'hsv_v\']]\n    hue,sat,val=cv2.split(cv2.cvtColor(np.asarray(image),cv2.COLOR_RGB2HSV))\n    x=np.arange(256,dtype=np.float32)\n    lut_h=((x+gains[0]*180)%180).astype(np.uint8)\n    lut_s=np.clip(x*(1+gains[1]),0,255).astype(np.uint8);lut_s[0]=0\n    lut_v=np.clip(x*(1+gains[2]),0,255).astype(np.uint8)\n    hsv=cv2.merge((cv2.LUT(hue,lut_h),cv2.LUT(sat,lut_s),cv2.LUT(val,lut_v)))\n    return Image.fromarray(cv2.cvtColor(hsv,cv2.COLOR_HSV2RGB))\n\nclass CocoInstances(Dataset):\n    def __init__(self,json_path,regime=\'none\',training=False):\n        self.raw=read_json(json_path);self.images=self.raw[\'images\'];self.training=training;self.regime=regime\n        self.by=defaultdict(list)\n        for a in self.raw[\'annotations\']:\n            if not a[\'iscrowd\']: self.by[a[\'image_id\']].append(a)\n    def __len__(self): return len(self.images)\n    def __getitem__(self,i):\n        from torchvision.transforms import functional as F, InterpolationMode\n        rec=self.images[i]; im=Image.open(rec[\'file_name\']).convert(\'RGB\'); w,h=im.size\n        anns=self.by[rec[\'id\']]\n        masks=np.stack([decode_segmentation(a[\'segmentation\'],h,w) for a in anns]) if anns else np.zeros((0,h,w),np.uint8)\n        masks=torch.from_numpy(masks); labels=torch.tensor([a[\'category_id\'] for a in anns],dtype=torch.int64)\n        if self.training:\n            if self.regime == \'default\' and random.random()<.5:\n                im=F.hflip(im);masks=masks.flip(-1)\n            if self.regime==\'custom\':\n                cfg=YOLO_AUG[\'custom\']\n                angle=random.uniform(-cfg[\'degrees\'],cfg[\'degrees\'])\n                shift=[round(random.uniform(-cfg[\'translate\'],cfg[\'translate\'])*w),\n                       round(random.uniform(-cfg[\'translate\'],cfg[\'translate\'])*h)]\n                zoom=random.uniform(1-cfg[\'scale\'],1+cfg[\'scale\'])\n                # Identical affine geometry for the image and every instance mask.\n                im=F.affine(im,angle,shift,zoom,[0.,0.],InterpolationMode.BILINEAR,fill=114)\n                if len(masks):\n                    masks=F.affine(masks,angle,shift,zoom,[0.,0.],InterpolationMode.NEAREST,fill=0)\n                im=custom_hsv(im,cfg)\n        # Recompute boxes from transformed masks (max coordinate +1, including one-pixel objects).\n        boxes=[]; kept=[]\n        for j,m in enumerate(masks):\n            ys,xs=torch.where(m>0)\n            if len(xs): boxes.append([xs.min(),ys.min(),xs.max()+1,ys.max()+1]); kept.append(j)\n        masks=masks[kept];labels=labels[kept]\n        boxes=torch.tensor(boxes,dtype=torch.float32).reshape(-1,4)\n        target=dict(boxes=boxes,labels=labels,masks=masks,image_id=torch.tensor(rec[\'id\']),\n                    area=masks.sum((1,2)).float(),iscrowd=torch.zeros(len(kept),dtype=torch.int64))\n        return F.to_tensor(im),target\n\ndef collate(batch): return tuple(zip(*batch))\n\ndef preview_maskrcnn_dataset(dataset, classes, output_dir, limit=6):\n    """Save transformed image/mask pairs as real augmentation-process evidence."""\n    from torchvision.transforms.functional import to_pil_image\n    from pycocotools import mask as mu\n    out=Path(output_dir);out.mkdir(parents=True,exist_ok=True)\n    for i in range(min(limit,len(dataset))):\n        image,target=dataset[i];path=out/f\'{i:02d}_image.jpg\';to_pil_image(image).save(path)\n        rec=dict(file_name=str(path),height=image.shape[1],width=image.shape[2])\n        preds=[]\n        for mask,label in zip(target[\'masks\'],target[\'labels\']):\n            rle=encode_mask(mask.numpy())\n            preds.append(dict(category_id=int(label),score=1.,segmentation=rle,bbox=mu.toBbox(rle).tolist()))\n        plot_predictions(rec,preds,{i+1:n for i,n in enumerate(classes)},out/f\'{i:02d}_ground_truth_masks.jpg\')\n\ndef plot_maskrcnn_history(run):\n    import matplotlib.pyplot as plt\n    with open(Path(run)/\'results.csv\',newline=\'\') as f: rows=list(csv.DictReader(f))\n    if not rows:return\n    x=[int(r[\'epoch\']) for r in rows]\n    fig,axes=plt.subplots(1,2,figsize=(10,4))\n    axes[0].plot(x,[float(r[\'train_loss\']) for r in rows]);axes[0].set_title(\'Training loss\')\n    axes[1].plot(x,[float(r[\'val_mask_AP50_95\']) for r in rows],label=\'mask AP50–95\')\n    axes[1].plot(x,[float(r[\'val_mask_AP50\']) for r in rows],label=\'mask AP50\');axes[1].legend()\n    axes[1].set_title(\'Original validation\');axes[1].set_ylim(0,1)\n    for ax in axes:ax.set_xlabel(\'Epoch\');ax.grid(alpha=.2)\n    fig.tight_layout();fig.savefig(Path(run)/\'training_curves.png\',dpi=150);plt.close(fig)\n\ndef build_maskrcnn(nclasses,imgsz=640,pretrained=True):\n    from torchvision.models.detection import maskrcnn_resnet50_fpn,MaskRCNN_ResNet50_FPN_Weights\n    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor\n    from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor\n    # Aspect ratio preserved: shorter side targets imgsz, longer side capped at imgsz.\n    model=maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT if pretrained else None,\n        weights_backbone=None,min_size=imgsz,max_size=imgsz,box_score_thresh=.001,box_detections_per_img=100)\n    model.roi_heads.box_predictor=FastRCNNPredictor(model.roi_heads.box_predictor.cls_score.in_features,nclasses+1)\n    model.roi_heads.mask_predictor=MaskRCNNPredictor(model.roi_heads.mask_predictor.conv5_mask.in_channels,256,nclasses+1)\n    return model\n\n@torch.inference_mode()\ndef predict_maskrcnn(model,json_path,device=\'cuda\'):\n    from pycocotools import mask as mu\n    model.eval();predictions=[]\n    ds=CocoInstances(json_path)\n    for image,target in ds:\n        pred=model([image.to(device)])[0]\n        for m,score,label in zip(pred[\'masks\'][:,0].cpu().numpy(),pred[\'scores\'].cpu().numpy(),pred[\'labels\'].cpu().numpy()):\n            if score<.001: continue\n            enc=encode_mask(m>=.5)\n            predictions.append(dict(image_id=int(target[\'image_id\']),category_id=int(label),score=float(score),\n                                    segmentation=enc,bbox=mu.toBbox(enc).tolist()))\n    return predictions\n\ndef save_checkpoint(path,state):\n    tmp=Path(str(path)+\'.tmp\');torch.save(state,tmp);tmp.replace(path)\n\ndef train_maskrcnn(prepared,run,regime,epochs=30,imgsz=640,batch=2,workers=2,lr=.0025):\n    seed_all();device=\'cuda\';model=build_maskrcnn(len(prepared[\'names\']),imgsz).to(device)\n    optimizer=torch.optim.SGD([p for p in model.parameters() if p.requires_grad],lr=lr,momentum=.9,weight_decay=.0005)\n    scheduler=torch.optim.lr_scheduler.MultiStepLR(optimizer,milestones=sorted(set([int(epochs*.67),int(epochs*.89)])),gamma=.1)\n    scaler=torch.amp.GradScaler(\'cuda\'); generator=torch.Generator().manual_seed(SEED)\n    ds=CocoInstances(prepared[\'paths\'][\'train\'],regime,True)\n    if not (run/\'augmentation_previews\').exists():\n        preview_maskrcnn_dataset(ds,prepared[\'names\'],run/\'augmentation_previews\')\n    seed_all()  # Preview sampling must not shift the training RNG stream, even after a retry.\n    loader=DataLoader(ds,batch_size=batch,shuffle=True,num_workers=workers,collate_fn=collate,\n                      pin_memory=True,worker_init_fn=seed_worker,generator=generator,persistent_workers=False)\n    start=0; best=-1.; elapsed=0.; last=run/\'last.pth\'\n    if last.exists():\n        ck=torch.load(last,map_location=\'cpu\',weights_only=False)  # only your own trusted checkpoint\n        model.load_state_dict(ck[\'model\']);optimizer.load_state_dict(ck[\'optimizer\']);scheduler.load_state_dict(ck[\'scheduler\'])\n        scaler.load_state_dict(ck[\'scaler\']);start=ck[\'epoch\'];best=ck[\'best_AP\'];elapsed=ck[\'train_val_seconds\']\n        random.setstate(ck[\'rng_python\']);np.random.set_state(ck[\'rng_numpy\']);torch.set_rng_state(ck[\'rng_torch\'])\n        torch.cuda.set_rng_state_all(ck[\'rng_cuda\']);generator.set_state(ck[\'rng_loader\'])\n    logpath=run/\'results.csv\'\n    # Remove any log row written after the latest successfully committed checkpoint.\n    if logpath.exists():\n        with open(logpath,newline=\'\') as f: rows=list(csv.DictReader(f));fields=list(rows[0]) if rows else []\n        if fields:\n            with open(logpath,\'w\',newline=\'\') as f:\n                writer=csv.DictWriter(f,fields);writer.writeheader();writer.writerows(r for r in rows if int(r[\'epoch\'])<=start)\n    for epoch in range(start,epochs):\n        torch.cuda.synchronize();t=time.perf_counter();model.train();totals=defaultdict(float)\n        for images,targets in loader:\n            images=[x.to(device,non_blocking=True) for x in images]\n            targets=[{k:v.to(device) for k,v in target.items()} for target in targets]\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=\'cuda\',dtype=torch.float16):\n                losses=model(images,targets);loss=sum(losses.values())\n            if not torch.isfinite(loss): raise FloatingPointError(f\'Nonfinite training loss at epoch {epoch+1}\')\n            scaler.scale(loss).backward();scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(model.parameters(),10.);scaler.step(optimizer);scaler.update()\n            for k,v in losses.items(): totals[k]+=float(v.detach())\n            totals[\'train_loss\']+=float(loss.detach())\n        learning_rate=optimizer.param_groups[0][\'lr\'];scheduler.step()\n        metrics=evaluate_coco(prepared[\'paths\'][\'valid\'],predict_maskrcnn(model,prepared[\'paths\'][\'valid\']),run/\'validation_latest\',0)\n        torch.cuda.synchronize();seconds=time.perf_counter()-t;elapsed+=seconds\n        row=dict(epoch=epoch+1,**{k:v/len(loader) for k,v in totals.items()},\n                 val_mask_AP50_95=metrics[\'mask_AP50_95\'],val_mask_AP50=metrics[\'mask_AP50\'],lr=learning_rate,\n                 epoch_seconds=seconds,train_val_seconds=elapsed)\n        with open(logpath,\'a\',newline=\'\') as f:\n            writer=csv.DictWriter(f,fieldnames=list(row))\n            if f.tell()==0: writer.writeheader()\n            writer.writerow(row)\n        score=metrics[\'mask_AP50_95\']\n        if score>best:\n            best=score\n            save_checkpoint(run/\'best.pth\',dict(model=model.state_dict(),classes=prepared[\'names\'],\n                                               epoch=epoch+1,imgsz=imgsz,best_AP=best))\n        save_checkpoint(last,dict(model=model.state_dict(),optimizer=optimizer.state_dict(),scheduler=scheduler.state_dict(),\n            scaler=scaler.state_dict(),epoch=epoch+1,best_AP=best,train_val_seconds=elapsed,classes=prepared[\'names\'],imgsz=imgsz,\n            rng_python=random.getstate(),rng_numpy=np.random.get_state(),rng_torch=torch.get_rng_state(),\n            rng_cuda=torch.cuda.get_rng_state_all(),rng_loader=generator.get_state()))\n        print(f\'Epoch {epoch+1}/{epochs}: loss={row["train_loss"]:.4f}, val mask AP={score:.4f}, {seconds:.1f}s\')\n    del model;torch.cuda.empty_cache()\n    plot_maskrcnn_history(run)\n    return run/\'best.pth\',elapsed\n\ndef completed_yolo_fit_time(run, epochs):\n    """Recover a completed fit if interruption occurred before wrapper bookkeeping."""\n    run=Path(run);last=run/\'fit/weights/last.pt\';best=run/\'fit/weights/best.pt\';csvpath=run/\'fit/results.csv\'\n    if not all(p.is_file() for p in (last,best,csvpath)): return None\n    with open(csvpath,newline=\'\') as f:\n        rows=[{k.strip():v for k,v in r.items()} for r in csv.DictReader(f)]\n    if not rows or int(float(rows[-1][\'epoch\']))<epochs: return None\n    ck=torch.load(last,map_location=\'cpu\',weights_only=False)  # own trusted checkpoint only\n    epoch=ck.get(\'epoch\',-2)\n    committed=epoch>=epochs-1 or (epoch==-1 and ck.get(\'optimizer\') is None)\n    return float(rows[-1][\'time\']) if committed else None\n\ndef train_yolo(prepared,run,model_name,regime,epochs=30,imgsz=640,batch=4,workers=2,lr=.001):\n    import ultralytics\n    from ultralytics import YOLO\n    if ultralytics.__version__!=\'8.3.221\': raise RuntimeError(\'Install the pinned ultralytics==8.3.221 version.\')\n    data=coco_to_yolo(prepared);seed_all();last=run/\'fit/weights/last.pt\'\n    # A completed fit can be evaluated again without asking YOLO to resume a stripped checkpoint.\n    fitdone=run/\'fit_complete.json\'\n    if fitdone.exists(): return run/\'fit/weights/best.pt\',read_json(fitdone)[\'train_val_seconds\']\n    recovered_time=completed_yolo_fit_time(run,epochs)\n    if recovered_time is not None:\n        write_json(fitdone,dict(train_val_seconds=recovered_time,recovered_completed_fit=True))\n        return run/\'fit/weights/best.pt\',recovered_time\n    kwargs=dict(data=data,epochs=epochs,imgsz=imgsz,batch=batch,device=0,workers=workers,\n                optimizer=\'AdamW\',lr0=lr,lrf=.01,weight_decay=.0005,seed=SEED,deterministic=True,\n                patience=0,amp=True,cache=False,save=True,save_period=5,plots=True,\n                project=str(run),name=\'fit\',exist_ok=True,pretrained=True,\n                multi_scale=False,rect=False,**YOLO_AUG[regime])\n    # Keep augmentation policy active during dataset construction, including a resumed run.\n    with yolo_augmentation_policy(regime):\n        model=YOLO(str(last) if last.exists() else model_name+\'.pt\')\n        torch.cuda.synchronize();t=time.perf_counter()\n        if last.exists(): model.train(resume=True,device=0,workers=workers)\n        else: model.train(**kwargs)\n        torch.cuda.synchronize();session_seconds=time.perf_counter()-t\n    # Native CSV time is cumulative within YOLO\'s recorded fit (including epoch validation).\n    with open(run/\'fit/results.csv\',newline=\'\') as f: rows=[{k.strip():v for k,v in r.items()} for r in csv.DictReader(f)]\n    native_seconds=float(rows[-1].get(\'time\',session_seconds))\n    write_json(fitdone,dict(train_val_seconds=native_seconds,current_session_seconds=session_seconds,\n                          note=\'Native results.csv time; interrupted YOLO sessions may reset its time column. See session ledger.\'))\n    del model;torch.cuda.empty_cache()\n    return run/\'fit/weights/best.pt\',native_seconds\n\ndef run_experiment(prepared,out_root,model_name,regime,epochs=30,imgsz=640,\n                   yolo_batch=4,mask_batch=2,workers=2):\n    require_gpu()\n    if model_name not in MODELS or regime not in REGIMES: raise ValueError(\'Unknown model/augmentation policy.\')\n    if not 1<=epochs<=1000: raise ValueError(\'Invalid epoch count.\')\n    run=Path(out_root)/f\'{model_name}_{regime}\';run.mkdir(parents=True,exist_ok=True)\n    config=dict(model=model_name,regime=regime,epochs=epochs,imgsz=imgsz,seed=SEED,\n                batch=mask_batch if model_name==\'maskrcnn\' else yolo_batch,\n                dataset_fingerprint=prepared[\'fingerprint\'],classes=prepared[\'names\'],\n                augmentation=YOLO_AUG[regime] if model_name!=\'maskrcnn\' or regime==\'custom\' else regime,\n                selection=\'Original validation set only; no test-based checkpoint selection\')\n    if (run/\'config.json\').exists() and read_json(run/\'config.json\')!=config:\n        raise ValueError(f\'Run settings changed: {run}. Use a new OUT_ROOT instead of mixing incompatible experiments.\')\n    write_json(run/\'config.json\',config)\n    for filename in (\'dataset_report.json\',\'classes.json\'):\n        shutil.copy2(Path(prepared[\'root\'])/filename,run/filename)\n    if (run/\'completed.json\').exists():\n        print(\'Already complete:\',run.name);return read_json(run/\'completed.json\')\n    collect_environment(run);session_start=time.time();status=\'interrupted_or_failed\'\n    try:\n        if model_name==\'maskrcnn\':\n            weights,seconds=train_maskrcnn(prepared,run,regime,epochs,imgsz,mask_batch,workers)\n            ck=torch.load(weights,map_location=\'cpu\',weights_only=False)\n            model=build_maskrcnn(len(prepared[\'names\']),imgsz,False).cuda();model.load_state_dict(ck[\'model\'])\n            t=time.perf_counter();pred=predict_maskrcnn(model,prepared[\'paths\'][\'test\'])\n            del model\n        else:\n            weights,seconds=train_yolo(prepared,run,model_name,regime,epochs,imgsz,yolo_batch,workers)\n            t=time.perf_counter();pred=predict_yolo(weights,prepared[\'paths\'][\'test\'],imgsz)\n        metrics=evaluate_coco(prepared[\'paths\'][\'test\'],pred,run/\'original_test\')\n        result=dict(model=model_name,regime=regime,epochs=epochs,weights=str(weights),\n                    train_val_seconds=seconds,test_seconds=time.perf_counter()-t,**metrics)\n        write_json(run/\'completed.json\',result);status=\'complete\'\n        return result\n    finally:\n        torch.cuda.empty_cache()\n        with open(run/\'session_ledger.jsonl\',\'a\') as f:\n            f.write(json.dumps(dict(start_unix=session_start,end_unix=time.time(),\n                allocated_session_seconds=time.time()-session_start,gpu_count=1,status=status))+\'\\n\')\n        # Ledger is wall time while a GPU is allocated, not measured kernel-only GPU time.\n\ndef prepare_own_test(json_path,run_dir,out_dir):\n    names=read_json(Path(run_dir)/\'classes.json\');canonical,hashes=canonicalize(json_path,names)\n    if len(canonical[\'images\'])<40: print(\'WARNING: assignment requires at least 40 own photos; this export has\',len(canonical[\'images\']))\n    out=Path(out_dir);out.mkdir(parents=True,exist_ok=True);p=out/\'own_test_canonical.json\';write_json(p,canonical)\n    write_json(out/\'own_test_manifest.json\',dict(source=str(json_path),source_sha256=sha256(json_path),image_hashes=hashes,\n                                               classes=names,note=\'Never used in training or checkpoint selection.\'))\n    return p\n\ndef evaluate_own(run_dir,own_json,out_dir=None,imgsz_override=None):\n    require_gpu();run=Path(run_dir);cfg=read_json(run/\'config.json\')\n    out=Path(out_dir) if out_dir else Path(\'/kaggle/working/own_evaluation\')/run.name\n    json_path=prepare_own_test(own_json,run,out);size=imgsz_override or cfg[\'imgsz\']\n    if cfg[\'model\']==\'maskrcnn\':\n        ck=torch.load(run/\'best.pth\',map_location=\'cpu\',weights_only=False)\n        if ck[\'classes\']!=cfg[\'classes\']: raise ValueError(\'Checkpoint class mapping mismatch.\')\n        model=build_maskrcnn(len(cfg[\'classes\']),size,False).cuda();model.load_state_dict(ck[\'model\'])\n        pred=predict_maskrcnn(model,json_path);del model\n    else: pred=predict_yolo(run/\'fit/weights/best.pt\',json_path,size)\n    metrics=evaluate_coco(json_path,pred,out,visual_limit=100000)\n    if (run/\'original_test/metrics.json\').exists() and not imgsz_override:\n        original=read_json(run/\'original_test/metrics.json\')\n        write_json(out/\'domain_shift.json\',dict(original_AP=original[\'mask_AP50_95\'],own_AP=metrics[\'mask_AP50_95\'],\n            drop_percentage_points=100*(original[\'mask_AP50_95\']-metrics[\'mask_AP50_95\'])))\n    write_json(out/\'evaluation_config.json\',dict(run=str(run),imgsz=size,diagnostic_override=bool(imgsz_override),\n        warning=\'A changed input size on own test is a post-hoc diagnostic, not the primary held-out score.\' if imgsz_override else \'\'))\n    torch.cuda.empty_cache();return metrics\n\ndef comparison_csv(out_root):\n    rows=[]\n    for p in sorted(Path(out_root).glob(\'*/completed.json\')):\n        d=read_json(p);rows.append({k:d[k] for k in (\'model\',\'regime\',\'epochs\',\'mask_AP50_95\',\'mask_AP50\',\'mask_AP75\',\'train_val_seconds\',\'test_seconds\')})\n    if not rows: return None\n    dest=Path(out_root)/\'comparison_original_test.csv\'\n    with open(dest,\'w\',newline=\'\') as f:\n        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)\n    return dest\n\ndef archive_run(run_dir,destination):\n    run=Path(run_dir);destination=Path(destination);destination.parent.mkdir(parents=True,exist_ok=True)\n    return shutil.make_archive(str(destination.with_suffix(\'\')),\'zip\',root_dir=run.parent,base_dir=run.name)\n', encoding="utf-8")
(CODE_DIR / "reference_classes.txt").write_text('backpack\t\n\t\n\tbed\t\n\t\n\tblanket\t\n\t\n\tbook\t\n\t\n\tbottle\t\n\t\n\tbowl\t\n\t\n\tbox\t\n\t\n\tcabinet-merged\t\n\t\n\tcardboard\t\n\t\n\tceiling-merged\t\n\t\n\tcell phone\t\n\t\n\tchair\t\n\t\n\tclock\t\n\t\n\tcouch\t\n\t\n\tcounter\t\n\t\n\tcup\t\n\t\n\tcurtain\t\n\t\n\tdining table\t\n\t\n\tdoor-stuff\t\n\t\n\tfire-extinguisher\t\n\t\n\tfloor-other-merged\t\n\t\n\tfloor-wood\t\n\t\n\tflower\t\n\t\n\tfood-other-merged\t\n\t\n\tfork\t\n\t\n\thair drier\t\n\t\n\thandbag\t\n\t\n\tkeyboard\t\n\t\n\tknife\t\n\t\n\tlaptop\t\n\t\n\tlight\t\n\t\n\tmicrowave\t\n\t\n\tmirror-stuff\t\n\t\n\tmouse\t\n\t\n\toven\t\n\t\n\tpan\t\n\t\n\tpaper\t\n\t\n\tpaper-merged\t\n\t\n\tperson\t\n\t\n\tpillow\t\n\t\n\tplate\t\n\t\n\tplug\t\n\t\n\tpotted plant\t\n\t\n\trefrigerator\t\n\t\n\tremote\t\n\t\n\trug-merged\t\n\t\n\tscissor\t\n\t\n\tscissors\t\n\t\n\tshelf\t\n\t\n\tsink\t\n\t\n\tspoon\t\n\t\n\tst\\\t\n\t\n\tstairs\t\n\t\n\tstove-merged\t\n\t\n\ttable-merged\t\n\t\n\tteddy bear\t\n\t\n\ttoaster\t\n\t\n\ttoilet\t\n\t\n\ttoothbrush\t\n\t\n\ttowel\t\n\t\n\ttoy\t\n\t\n\ttrash-bin\t\n\t\n\ttree-merged\t\n\t\n\ttv\t\n\t\n\tvase\t\n\t\n\twall\t\n\t\n\twall-brick\t\n\t\n\twall-other-merged\t\n\t\n\twall-stone\t\n\t\n\twall-tile\t\n\t\n\twall-wood\t\n\t\n\twallet\t\n\t\n\twindow-blind\t\n\t\n\twindow-other', encoding="utf-8")
if str(CODE_DIR) not in sys.path: sys.path.insert(0, str(CODE_DIR))
import seg046
s = importlib.reload(seg046)
print("Training/evaluation helpers loaded.")


In [ ]:
# Mask R-CNN: expand one mask at a time on CPU, then immediately encode it.
# Restart the Kaggle session after an OOM before running this notebook.
import gc
import torch
from PIL import Image
from torchvision.transforms import functional as vision_F


def maskrcnn_cpu_postprocess(original_postprocess, result, image_shapes, original_image_sizes):
    processed = []
    for prediction, image_shape, original_size in zip(result, image_shapes, original_image_sizes):
        cpu = {key: value.detach().cpu() for key, value in prediction.items()}
        encoded_masks, boxes = [], []
        for index in range(len(cpu["scores"])):
            one = {key: value[index:index + 1] for key, value in cpu.items()}
            # Use the installed torchvision's own box scaling and mask interpolation.
            full = original_postprocess([one], [image_shape], [original_size])[0]
            mask = full["masks"][0, 0].numpy() >= 0.5
            encoded_masks.append(s.encode_mask(mask))
            boxes.append(full["boxes"])
            del mask, full, one
        cpu.pop("masks", None)  # Discard the small ROI masks as well.
        if boxes:
            cpu["boxes"] = torch.cat(boxes, dim=0)
        cpu["encoded_masks"] = encoded_masks
        processed.append(cpu)
    return processed


@torch.inference_mode()
def predict_maskrcnn_low_memory(model, json_path, device="cuda"):
    from pycocotools import mask as mask_utils
    model.eval()
    predictions = []
    original_postprocess = model.transform.postprocess

    def cpu_postprocess(result, image_shapes, original_image_sizes):
        return maskrcnn_cpu_postprocess(original_postprocess, result, image_shapes, original_image_sizes)

    model.transform.postprocess = cpu_postprocess
    image = pred = None
    try:
        records = s.read_json(json_path)["images"]
        for index, rec in enumerate(records, 1):
            print(f"Mask R-CNN {index}/{len(records)}: {Path(rec['file_name']).name}", flush=True)
            # Inference needs only pixels; do not decode/stack ground-truth masks here.
            with Image.open(rec["file_name"]) as im:
                image = vision_F.to_tensor(im.convert("RGB"))
            pred = model([image.to(device)])[0]
            for encoded, score, label in zip(pred["encoded_masks"], pred["scores"].tolist(), pred["labels"].tolist()):
                if score < .001:
                    continue
                predictions.append(dict(
                    image_id=rec["id"], category_id=int(label), score=float(score),
                    segmentation=encoded, bbox=mask_utils.toBbox(encoded).tolist()
                ))
            image = pred = None
    finally:
        model.transform.postprocess = original_postprocess
        image = pred = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return predictions


s.predict_maskrcnn = predict_maskrcnn_low_memory
print("Mask R-CNN low-memory evaluation enabled: GPU inference, one full-size mask at a time on CPU.")
print("Input resolution, confidence threshold, detection limit and held-out annotations remain unchanged.")


In [ ]:
# Pinned Ultralytics 8.3.221: retain GPU inference and decode masks one at a time on CPU.
import gc
import numpy as np
import torch
import ultralytics
from ultralytics import YOLO
from ultralytics.models.yolo.segment.predict import SegmentationPredictor
from ultralytics.engine.results import Results
from ultralytics.utils import ops

if ultralytics.__version__ != "8.3.221":
    raise RuntimeError("Run the install cell and restart the kernel: this patch targets ultralytics==8.3.221.")


def encode_yolo_native_one_by_one(proto, coefficients, boxes, original_shape):
    count = len(coefficients)
    if count == 0:
        return [], []
    channels, mh, mw = proto.shape
    # Same low-resolution matrix multiplication as the original GPU code.
    logits = (coefficients @ proto.float().view(channels, -1)).view(-1, mh, mw).detach().cpu()
    boxes = boxes.detach().cpu()
    height, width = original_shape
    encoded, kept = [], []
    for index in range(count):
        mask = ops.scale_masks(logits[index:index + 1][None], original_shape)[0]
        # 8.3.221 has different cropping rules below/above 50 detections.
        # Preserve the branch selected by the ORIGINAL count, not the chunk size.
        if count < 50:
            mask = ops.crop_mask(mask, boxes[index:index + 1])
        else:
            x1, y1, x2, y2 = boxes[index]
            rows = torch.arange(height, dtype=boxes.dtype)[:, None]
            cols = torch.arange(width, dtype=boxes.dtype)[None, :]
            mask *= ((cols >= x1) & (cols < x2) & (rows >= y1) & (rows < y2))[None]
        binary = mask[0] > 0.0
        if bool(binary.any()):
            encoded.append(s.encode_mask(binary.numpy()))
            kept.append(index)
        del mask, binary
    return encoded, kept


class LowMemorySegmentationPredictor(SegmentationPredictor):
    def construct_result(self, pred, img, orig_img, img_path, proto):
        if not self.args.retina_masks:
            raise ValueError("Low-memory evaluation expects retina_masks=True.")
        if len(pred):
            pred[:, :4] = ops.scale_boxes(img.shape[2:], pred[:, :4], orig_img.shape)
            encoded, kept = encode_yolo_native_one_by_one(proto, pred[:, 6:], pred[:, :4], orig_img.shape[:2])
            boxes = pred[:, :6].detach().cpu()[torch.tensor(kept, dtype=torch.long)]
        else:
            encoded = []
            boxes = pred[:, :6].detach().cpu()
        result = Results(orig_img, path=img_path, names=self.model.names, boxes=boxes, masks=None)
        # Compressed masks replace the huge dense mask stack for this evaluator.
        result.encoded_masks = encoded
        return result


@torch.inference_mode()
def predict_yolo_low_memory(weights, json_path, imgsz=640):
    from pycocotools import mask as mask_utils
    model = YOLO(str(weights))
    predictions = []
    result = None
    try:
        records = s.read_json(json_path)["images"]
        for index, rec in enumerate(records, 1):
            print(f"YOLO {index}/{len(records)}: {Path(rec['file_name']).name}", flush=True)
            result = model.predict(
                rec["file_name"], predictor=LowMemorySegmentationPredictor,
                imgsz=imgsz, device=0, batch=1, conf=.001, iou=.7,
                max_det=100, retina_masks=True, verbose=False, save=False
            )[0]
            for encoded, score, cls in zip(result.encoded_masks, result.boxes.conf.tolist(), result.boxes.cls.tolist()):
                if tuple(encoded["size"]) != (rec["height"], rec["width"]):
                    raise RuntimeError("YOLO mask dimensions differ from the source image.")
                predictions.append(dict(
                    image_id=rec["id"], category_id=int(cls) + 1, score=float(score),
                    segmentation=encoded, bbox=mask_utils.toBbox(encoded).tolist()
                ))
            result = None
            # Ultralytics retains the last Results object; ours contains CPU RLE only.
            model.predictor.results = []
    finally:
        result = None
        if model.predictor is not None:
            model.predictor.results = []
        model.predictor = None
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return predictions


s.predict_yolo = predict_yolo_low_memory
print("YOLO low-memory evaluation enabled: GPU inference, native-size masks encoded one at a time on CPU.")


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import csv
import json
from PIL import Image


def missing_segmentation(seg):
    if seg is None:
        return True
    if isinstance(seg, list):
        return not seg or all(isinstance(poly, list) and not poly for poly in seg)
    if isinstance(seg, dict):
        return seg.get("counts") is None or seg.get("counts") == [] or seg.get("counts") == ""
    return False  # Other malformed values remain errors in the strict evaluator.


def filter_own_annotations(source_json, output_dir, skip_size=True, skip_missing=True):
    source_json = Path(source_json)
    if not source_json.is_file():
        raise FileNotFoundError(f"COCO annotation JSON not found: {source_json}")
    raw = json.loads(source_json.read_text(encoding="utf-8"))
    image_ids = [rec["id"] for rec in raw["images"]]
    if len(image_ids) != len(set(image_ids)):
        raise ValueError("Duplicate image IDs in source COCO JSON. Repair the source export.")
    known_ids = set(image_ids)
    if any(ann["image_id"] not in known_ids for ann in raw["annotations"]):
        raise ValueError("Source annotation references an absent image. Repair the source export.")
    by_image = defaultdict(list)
    for ann in raw["annotations"]:
        by_image[ann["image_id"]].append(ann)
    category_names = {cat["id"]: cat["name"] for cat in raw["categories"]}
    kept, skipped, blocked = [], [], False
    for rec in raw["images"]:
        path = source_json.parent / rec["file_name"]
        if not path.is_file():
            path = source_json.parent / Path(rec["file_name"]).name
        if not path.is_file():
            raise FileNotFoundError(f"Missing image: {rec['file_name']}")
        with Image.open(path) as im:
            width, height = im.size
        reasons = []
        if (width, height) != (rec["width"], rec["height"]):
            reasons.append("image_size_mismatch")
            blocked = blocked or not skip_size
        bad = [ann for ann in by_image[rec["id"]] if missing_segmentation(ann.get("segmentation"))]
        if bad:
            reasons.append("missing_or_empty_segmentation")
            blocked = blocked or not skip_missing
        if reasons:
            skipped.append({
                "image_id": rec["id"], "file_name": rec["file_name"],
                "image_path": str(path.resolve()), "reason": "; ".join(reasons),
                "metadata_width": rec["width"], "metadata_height": rec["height"],
                "actual_width": width, "actual_height": height,
                "excluded_annotations": len(by_image[rec["id"]]),
                "missing_mask_annotation_ids": [ann.get("id") for ann in bad],
                "missing_mask_classes": [category_names.get(ann["category_id"], str(ann["category_id"])) for ann in bad],
            })
        else:
            kept.append({**rec, "file_name": str(path.resolve())})
    kept_ids = {rec["id"] for rec in kept}
    annotations = [ann for ann in raw["annotations"] if ann["image_id"] in kept_ids]
    report = {
        "source_json": str(source_json.resolve()),
        "input_images": len(raw["images"]), "retained_images": len(kept),
        "skipped_images": len(skipped),
        "input_annotations": len(raw["annotations"]),
        "retained_annotations": len(annotations),
        "excluded_annotations": len(raw["annotations"]) - len(annotations),
        "reason_counts": dict(Counter(reason for item in skipped for reason in item["reason"].split("; "))),
        "skip_size_enabled": bool(skip_size), "skip_missing_enabled": bool(skip_missing),
        "evaluation_blocked": bool(blocked), "skipped": skipped,
        "note": "Metrics apply only to retained images. Entire images excluded; no image resizing or polygon edits. Reason counts can overlap.",
    }
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    report_path = output_dir / "skip_report.json"
    csv_path = output_dir / "skipped_images.csv"
    report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    fields = ["image_id", "file_name", "image_path", "reason", "metadata_width", "metadata_height",
              "actual_width", "actual_height", "excluded_annotations", "missing_mask_annotation_ids", "missing_mask_classes"]
    with csv_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        for item in skipped:
            writer.writerow({k: json.dumps(v, ensure_ascii=False) if isinstance(v, list) else v for k, v in item.items()})
    print(f"Images: {len(raw['images'])} total | {len(kept)} retained | {len(skipped)} excluded")
    for item in skipped:
        action = "PROBLEM" if blocked else "SKIPPED"
        print(f"{action}: {item['file_name']} | {item['reason']}")
        if "image_size_mismatch" in item["reason"]:
            print(f"  COCO: {item['metadata_width']}x{item['metadata_height']} | actual: {item['actual_width']}x{item['actual_height']}")
        if item["missing_mask_annotation_ids"]:
            print(f"  Missing masks: annotation IDs {item['missing_mask_annotation_ids']} | classes {item['missing_mask_classes']}")
    print("CSV report:", csv_path)
    print("JSON report:", report_path)
    if blocked:
        raise ValueError("Preflight found problems with skipping disabled. See reports or enable the corresponding SKIP setting.")
    if not kept:
        raise ValueError("No images remain after checks. Reports were saved; repair/re-export the source annotations.")
    if not annotations:
        raise ValueError("No object annotations remain after checks. Reports were saved; repair/re-export the source annotations.")
    filtered_path = output_dir / "filtered_annotations.coco.json"
    filtered_path.write_text(json.dumps({**raw, "images": kept, "annotations": annotations}), encoding="utf-8")
    if skipped:
        print("WARNING: Evaluation covers a reduced subset; excluded images are not scored.")
    return str(filtered_path), report



print("Annotation checks ready; selected photo sets will be checked in the evaluation cell.")


In [ ]:
# Render readable, numbered object labels with matching legends and ground truth.
import csv
import colorsys
import re
from collections import defaultdict
from PIL import Image, ImageDraw, ImageFont, ImageFilter


def comparison_font(size=20):
    for name in ("DejaVuSans.ttf", "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
                 "/usr/share/fonts/truetype/liberation2/LiberationSans-Regular.ttf"):
        try:
            return ImageFont.truetype(name, size)
        except OSError:
            pass
    try:
        return ImageFont.load_default(size=size)
    except TypeError:
        return ImageFont.load_default()


def class_color(category_id):
    rgb = colorsys.hsv_to_rgb((int(category_id) * .61803398875) % 1, .72, .95)
    return tuple(round(v * 255) for v in rgb)


def draw_instance_panel(original, instances, categories, title, panel_width=900, predicted=False):
    scale = min(panel_width / original.width, 1100 / original.height)
    size = (max(1, round(original.width * scale)), max(1, round(original.height * scale)))
    photo = original.resize(size, Image.Resampling.LANCZOS).convert("RGBA")
    marks, legend = [], []
    font = comparison_font(20)
    for number, instance in enumerate(instances, 1):
        color = class_color(instance["category_id"])
        mask = s.decode_segmentation(instance["segmentation"], original.height, original.width)
        small_mask = Image.fromarray(mask.astype("uint8") * 255).resize(size, Image.Resampling.NEAREST)
        tint = Image.new("RGBA", size, color + (0,))
        tint.putalpha(small_mask.point(lambda x: round(x * .27)))
        photo = Image.alpha_composite(photo, tint)
        # Thin visible contour around each retained instance.
        edge = Image.fromarray(np.asarray(small_mask.filter(ImageFilter.MaxFilter(3))) - np.asarray(small_mask.filter(ImageFilter.MinFilter(3))))
        outline = Image.new("RGBA", size, color + (0,))
        outline.putalpha(edge)
        photo = Image.alpha_composite(photo, outline)
        bbox = small_mask.getbbox()
        name = categories[instance["category_id"]]
        label = f"#{number} {name}"
        if predicted:
            label += f" {instance['score']:.0%}"
        if bbox:
            marks.append((bbox[0], bbox[1], label, color))
        legend.append((label, color))
        del mask, small_mask, tint, edge, outline
    # Draw labels after masks so later masks cannot obscure text.
    draw = ImageDraw.Draw(photo)
    occupied = []
    for x, y, label, color in marks:
        width = min(size[0], int(draw.textlength(label, font=font)) + 14)
        height = 30
        x = max(0, min(x, size[0] - width))
        y = max(0, min(y, size[1] - height))
        for _ in range(12):
            if not any(x < bx2 and x + width > bx1 and y < by2 and y + height > by1 for bx1, by1, bx2, by2 in occupied):
                break
            next_y = y + height + 3
            if next_y > size[1] - height:
                break
            y = next_y
        draw.rectangle((x, y, x + width, y + height), fill=(10, 15, 24, 235), outline=color + (255,), width=2)
        draw.text((x + 6, y + 3), label, font=font, fill="white")
        occupied.append((x, y, x + width, y + height))
    # Two-column full legend remains readable even if in-image labels overlap.
    legend_rows = max(1, (len(legend) + 1) // 2)
    panel = Image.new("RGB", (panel_width, 62 + size[1] + 48 + legend_rows * 32), (18, 24, 34))
    draw = ImageDraw.Draw(panel)
    draw.text((18, 16), title, font=comparison_font(24), fill="white")
    panel.paste(photo.convert("RGB"), ((panel_width - size[0]) // 2, 62))
    top = 62 + size[1] + 12
    draw.text((18, top), f"{len(instances)} instances shown", font=font, fill=(190, 200, 215))
    for i, (label, color) in enumerate(legend):
        x = 18 + (i % 2) * (panel_width // 2)
        y = top + 34 + (i // 2) * 32
        draw.rectangle((x, y + 4, x + 14, y + 18), fill=color)
        draw.text((x + 23, y), label, font=font, fill="white")
    return panel


def render_run_comparisons(out_dir, threshold=.25):
    out_dir = Path(out_dir)
    raw = s.read_json(out_dir / "own_test_canonical.json")
    predictions = s.read_json(out_dir / "predictions_coco.json")
    categories = {cat["id"]: cat["name"] for cat in raw["categories"]}
    gt_by, pred_by = defaultdict(list), defaultdict(list)
    for ann in raw["annotations"]:
        gt_by[ann["image_id"]].append(ann)
    for pred in predictions:
        pred_by[pred["image_id"]].append(pred)
    folder = out_dir / "labeled_comparisons"
    folder.mkdir(parents=True, exist_ok=True)
    object_rows, image_rows, created = [], [], []
    for rec in raw["images"]:
        predicted = sorted(pred_by[rec["id"]], key=lambda p: p["score"], reverse=True)
        shown = [p for p in predicted if p["score"] >= threshold]
        truth = gt_by[rec["id"]]
        with Image.open(rec["file_name"]) as im:
            original = im.convert("RGB")
        left = draw_instance_panel(original, truth, categories, "YOUR ANNOTATIONS", predicted=False)
        right = draw_instance_panel(original, shown, categories, f"MODEL PREDICTIONS | confidence >= {threshold:.0%}", predicted=True)
        canvas = Image.new("RGB", (left.width + right.width + 16, max(left.height, right.height) + 84), (8, 12, 20))
        draw = ImageDraw.Draw(canvas)
        filename = Path(rec["file_name"]).name
        draw.text((18, 12), filename, font=comparison_font(18), fill="white")
        draw.text((18, 42), "Numbers identify objects within each panel; they are not matched pairs. Confidence cutoff affects display only.", font=comparison_font(17), fill=(190, 200, 215))
        canvas.paste(left, (0, 84))
        canvas.paste(right, (left.width + 16, 84))
        safe = re.sub(r"[^A-Za-z0-9_.-]", "_", Path(filename).stem)
        dest = folder / f"{rec['id']}_{safe}_comparison.jpg"
        canvas.save(dest, quality=92)
        created.append(dest)
        image_rows.append(dict(image_id=rec["id"], file_name=filename, ground_truth_instances=len(truth),
                               predictions_total=len(predicted), predictions_shown=len(shown), display_threshold=threshold,
                               comparison_file=dest.name))
        for kind, instances in (("ground_truth", truth), ("prediction", predicted)):
            for number, instance in enumerate(instances, 1):
                object_rows.append(dict(image_id=rec["id"], file_name=filename, source=kind,
                    instance_number=number, class_name=categories[instance["category_id"]],
                    confidence=instance.get("score", ""),
                    shown=True if kind == "ground_truth" else instance["score"] >= threshold))
    for name, rows, fields in (
        ("objects_and_classes.csv", object_rows, ["image_id", "file_name", "source", "instance_number", "class_name", "confidence", "shown"]),
        ("image_comparison_index.csv", image_rows, ["image_id", "file_name", "ground_truth_instances", "predictions_total", "predictions_shown", "display_threshold", "comparison_file"]),
    ):
        with (out_dir / name).open("w", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=fields)
            writer.writeheader()
            writer.writerows(rows)
    print("Labeled comparisons:", folder)
    return created


# Avoid generating the old tiny-label overlays. Metrics/prediction JSON are unchanged.
if not hasattr(s, "_evaluate_coco_before_readable_views"):
    s._evaluate_coco_before_readable_views = s.evaluate_coco

def evaluate_coco_without_old_overlays(json_path, predictions, out_dir, visual_limit=12):
    return s._evaluate_coco_before_readable_views(json_path, predictions, out_dir, visual_limit=0)

s.evaluate_coco = evaluate_coco_without_old_overlays
print("Readable side-by-side comparisons enabled. Class labels, confidence, mask outlines and complete legends included.")


In [ ]:
from IPython.display import display, FileLink, Image as DisplayImage
import pandas as pd
import shutil
import zipfile

MODEL_GROUPS = {
    "all": ("yolo11n-seg", "yolo11s-seg", "maskrcnn"),
    "yolo": ("yolo11n-seg", "yolo11s-seg"),
    "mask": ("maskrcnn",),
}
VALID_REGIMES = ("none", "default", "custom")


def resolve_selection(selection, augmentations, photo_sets):
    if isinstance(selection, str):
        group = selection.lower().strip()
        if group not in MODEL_GROUPS:
            raise ValueError('MODELS must be "all", "yolo", "mask", or a list such as ["yolo11n-seg"].')
        models = list(MODEL_GROUPS[group])
    elif isinstance(selection, (list, tuple)) and selection:
        models = list(dict.fromkeys(selection))
    else:
        raise ValueError('MODELS must be "all", "yolo", "mask", or a nonempty list of model names.')
    for name, value in (("AUGMENTATIONS", augmentations), ("PHOTO_SETS", photo_sets)):
        if not isinstance(value, (list, tuple)):
            raise ValueError(f"{name} must be a list, e.g. ['custom'], not a single string.")
    regimes = list(dict.fromkeys(augmentations))
    sets = list(dict.fromkeys(photo_sets))
    unknown_models = [name for name in models if name not in MODEL_GROUPS["all"]]
    if unknown_models:
        raise ValueError(f"Unknown model names: {unknown_models}. Valid names: {MODEL_GROUPS['all']}")
    if not regimes or any(regime not in VALID_REGIMES for regime in regimes):
        raise ValueError("AUGMENTATIONS must select one or more of: none, default, custom.")
    if not sets or any(name not in ("Normal", "Difficult") for name in sets):
        raise ValueError('PHOTO_SETS must contain "Normal", "Difficult", or both.')
    return models, regimes, sets


def find_selected_runs(root, models, regimes):
    expected = [(model, regime) for model in models for regime in regimes]
    found = {key: [] for key in expected}
    for config_path in sorted(Path(root).rglob("config.json")):
        if not (config_path.parent / "completed.json").is_file():
            continue
        cfg = s.read_json(config_path)
        key = (cfg.get("model"), cfg.get("regime"))
        if key in found:
            found[key].append(config_path.parent)
    missing = [f"{model}_{regime}" for (model, regime), folders in found.items() if not folders]
    duplicate = {f"{model}_{regime}": [str(folder) for folder in folders]
                 for (model, regime), folders in found.items() if len(folders) > 1}
    if missing:
        raise FileNotFoundError("Attach these selected completed training runs: " + ", ".join(missing))
    if duplicate:
        raise ValueError(f"Multiple copies of selected runs found. Keep one copy or narrow RUNS_ROOT: {duplicate}")
    runs = [found[key][0] for key in expected]
    for run in runs:
        cfg = s.read_json(run / "config.json")
        weights = run / ("best.pth" if cfg["model"] == "maskrcnn" else "fit/weights/best.pt")
        if not weights.is_file():
            raise FileNotFoundError(f"Selected run is missing its trained weights: {weights}")
    return runs


def matching_saved_evaluation(folder, run, cfg, weight_hash, source_hash, image_hashes):
    folder = Path(folder)
    required = ("evaluation_config.json", "own_test_manifest.json", "own_test_canonical.json", "metrics.json", "predictions_coco.json")
    if not all((folder / name).is_file() for name in required):
        return False
    metadata = s.read_json(folder / "evaluation_config.json")
    manifest = s.read_json(folder / "own_test_manifest.json")
    old_hash = metadata.get("checkpoint_sha256")
    checkpoint_matches = old_hash == weight_hash if old_hash else REUSE_LEGACY_RESULTS
    return bool(checkpoint_matches and metadata.get("run") == str(run)
        and metadata.get("imgsz") == cfg["imgsz"] and not metadata.get("diagnostic_override", False)
        and manifest.get("source_sha256") == source_hash and manifest.get("classes") == cfg["classes"]
        and manifest.get("image_hashes") == image_hashes)


def archive_selected_results(root, run_names, destination):
    """Package only the current selection, even if old unselected runs remain on disk."""
    root, destination = Path(root), Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    paths = [root / name for name in run_names] + [root / "preflight"]
    files = [root / "domain_shift_comparison.csv", root / "evaluation_selection.json"]
    with zipfile.ZipFile(destination, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for folder in paths:
            if folder.is_dir():
                for file in sorted(folder.rglob("*")):
                    if file.is_file():
                        archive.write(file, str(Path(root.name) / file.relative_to(root)))
        for file in files:
            if file.is_file():
                archive.write(file, str(Path(root.name) / file.name))
    return str(destination)


models, regimes, photo_sets = resolve_selection(MODELS, AUGMENTATIONS, PHOTO_SETS)
run_dirs = find_selected_runs(RUNS_ROOT, models, regimes)
print("SELECTED MODELS:", models)
print("SELECTED AUGMENTATIONS:", regimes)
print("SELECTED PHOTO SETS:", photo_sets)
print(f"{len(run_dirs)} model/checkpoint combinations x {len(photo_sets)} photo sets = {len(run_dirs) * len(photo_sets)} evaluation jobs.")
print("Only these selected jobs will execute. No training is performed.")
# Validate all chosen photo sets before starting any model inference.
jobs = []
for own_set in photo_sets:
    source = Path(OWN_DATA_ROOT) / f"SSWAFFLELOVER21-Folder- {own_set}.coco" / "test/_annotations.coco.json"
    root = Path(OUTPUT_BASE) / own_set.lower()
    print(f"\nChecking {own_set} annotations...")
    filtered, report = filter_own_annotations(source, root / "preflight", skip_size=SKIP_SIZE_MISMATCH, skip_missing=SKIP_MISSING_SEGMENTATION)
    display(FileLink(str(root / "preflight/skipped_images.csv")))
    s.write_json(root / "evaluation_selection.json", dict(models=models, augmentations=regimes, photo_set=own_set,
        display_confidence=DISPLAY_CONFIDENCE, selected_run_names=[run.name for run in run_dirs]))
    image_hashes = {str(rec["id"]): s.sha256(rec["file_name"]) for rec in s.read_json(filtered)["images"]}
    jobs.append(dict(name=own_set, root=root, filtered=filtered, report=report,
                     source_hash=s.sha256(filtered), image_hashes=image_hashes))
s.require_gpu()
attached_evaluations = sorted({p.parent for p in Path("/kaggle/input").rglob("own_test_manifest.json")}) if REUSE_EXISTING_RESULTS else []
weight_hashes = {}
all_rows = []
for job in jobs:
    own_set, root = job["name"], job["root"]
    rows = []
    for run in run_dirs:
        print(f"\n[{own_set}] Evaluating: {run.name}", flush=True)
        gc.collect()
        torch.cuda.empty_cache()
        cfg = s.read_json(run / "config.json")
        # Guard against a changed config selecting an unintended model after discovery.
        if cfg.get("model") not in models or cfg.get("regime") not in regimes:
            raise ValueError(f"Run settings changed after selection: {run}")
        weight_path = run / ("best.pth" if cfg["model"] == "maskrcnn" else "fit/weights/best.pt")
        if str(weight_path) not in weight_hashes:
            weight_hashes[str(weight_path)] = s.sha256(weight_path)
        weight_hash = weight_hashes[str(weight_path)]
        out = root / run.name
        candidates = [out, Path(f"/kaggle/working/own_evaluation_{own_set.lower()}") / run.name,
                      Path(f"/kaggle/working/yolo_evaluation_{own_set.lower()}") / run.name]
        candidates.extend(folder for folder in attached_evaluations if folder.name == run.name)
        cached = next((folder for folder in candidates if matching_saved_evaluation(
            folder, run, cfg, weight_hash, job["source_hash"], job["image_hashes"])), None) if REUSE_EXISTING_RESULTS else None
        if cached is not None:
            print("Reusing completed predictions and metrics:", cached)
            if cached != out:
                out.mkdir(parents=True, exist_ok=True)
                for file in cached.iterdir():
                    if file.is_file():
                        shutil.copy2(file, out / file.name)
            metrics = s.read_json(out / "metrics.json")
        else:
            metrics = s.evaluate_own(run, job["filtered"], out)
        eval_cfg = s.read_json(out / "evaluation_config.json")
        if cached is None:
            eval_cfg["mask_postprocess"] = "GPU network; CPU native-resolution masks, one at a time"
        eval_cfg["checkpoint_sha256"] = weight_hash
        eval_cfg["display_confidence"] = DISPLAY_CONFIDENCE
        s.write_json(out / "evaluation_config.json", eval_cfg)
        views = render_run_comparisons(out, threshold=DISPLAY_CONFIDENCE)
        if views:
            display(DisplayImage(filename=str(views[0]), width=1100))
        display(FileLink(str(out / "objects_and_classes.csv")))
        old = s.read_json(run / "original_test/metrics.json")
        row = dict(photo_set=own_set, model=cfg["model"], augmentation=cfg["regime"],
            own_images_evaluated=job["report"]["retained_images"], own_images_skipped=job["report"]["skipped_images"],
            original_mask_AP50_95=old["mask_AP50_95"], own_mask_AP50_95=metrics["mask_AP50_95"],
            drop_percentage_points=100 * (old["mask_AP50_95"] - metrics["mask_AP50_95"]))
        rows.append(row)
        all_rows.append(row)
        pd.DataFrame(rows).to_csv(root / "domain_shift_comparison.csv", index=False)
        pd.DataFrame(all_rows).to_csv(Path(OUTPUT_BASE) / "selected_evaluations.csv", index=False)
        archive = s.archive_run(out, f"/kaggle/working/{own_set.lower()}_{run.name}_evaluation.zip")
        display(FileLink(archive))
        gc.collect()
        torch.cuda.empty_cache()
    display(pd.DataFrame(rows))
    archive = archive_selected_results(root, [run.name for run in run_dirs], f"/kaggle/working/selected_models_evaluation_{own_set.lower()}.zip")
    display(FileLink(archive))
print("Completed the selected evaluations. Normal and Difficult scores are separate; their mAP values are not a pooled score.")
display(FileLink(str(Path(OUTPUT_BASE) / "selected_evaluations.csv")))
